# 6.1 Wearables Basics

**Wearables series, class 1.** Tens of thousands of *All of Us* participants share data from a **Fitbit** or from **Apple Health** (iPhone and Apple Watch): steps, heart rate, sleep, and more, often every minute, for years. That's a view of daily life that health records and surveys can't give. It also brings new problems: tables measured in **terabytes**, days when the device wasn't worn, two kinds of device that measure differently, and a group of people who are far from typical. In this class we learn where the wearable data is, what it costs to use, and how to turn it into one clean value per person.

### By the end of this notebook you'll be able to

1. Describe the Fitbit and Apple HealthKit data in *All of Us*: tables, layouts, and who has them
2. Estimate and check the **cost** of a query, and query huge **minute-level** tables safely
3. Define a **valid wear day**, and explain why it matters
4. Summarise many days into **one value per person**
5. Explain why step counts from a Fitbit, an iPhone, and an Apple Watch can't simply be pooled
6. Describe who has wearable data, and what that means for **generalisability**

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Who in our cohort has wearable data, how much, and how do we turn it into usable variables? |
| 🧪 **The analysis** | Table sizes and costs; who has data (cases vs controls, age, sex); valid wear days; median daily steps per person, by device; steps by hour of the day |
| 🎯 **Why** | Wearable data is easy to misuse: un-worn days look like inactive days, devices differ, and one careless query can cost tens of dollars |
| ✅ **What we conclude** | How many of our cases and controls have usable data, and a per-person step summary for **6.2** |
| ⚠️ **What it can't tell us** | Anything about cause. And people with wearables aren't typical: results describe **them** |
| 🔗 **Connections** | Cohort from **1.1**; units and cleaning from **2.1**; long vs wide from **2.2**; no survey weights from **5.1**; **6.2** and **6.3** use these files |

| | |
|---|---|
| 📂 **Uses** | the Controlled Tier dataset (Fitbit tables); the Apple HealthKit dataset; `cohort_sample.csv` (**1.1**; made for you if it's missing) |
| 💾 **Saves** | `fitbit_daily.csv` (one row per person per day), `wearables_person.csv` (one row per person) |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 20 |
| 2. Setup, and two cost helpers | 5 |
| 3. A tour of the tables, and their cost | 10 |
| 4. Fitbit: daily data, and valid days | 20 |
| 5. Minute-level data, safely | 10 |
| 6. Apple HealthKit | 15 |
| 7. One row per person; who has wearables? | 10 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 95 minutes |
| ⏱️ **Run time** | about **5 minutes** to run every cell |
| 💰 **Data cost** | about **$1.40** in our test run: $1.20 of it the Apple HealthKit query, the rest a few cents each. Every query is priced **before** it runs, and checked after |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| ⚠️ **Never** | query a minute-level table (`steps_intraday`, `heart_rate_intraday`) **without** a `person_id` filter: a full read is over **6 TB**, about **$40** |
| **Estimated total** | about **$2** per student for this class |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. A plot of **one person's** steps is participant-level data: plot groups. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### Two sources of wearable data

| | Fitbit | Apple HealthKit |
|---|---|---|
| **How it got here** | participants link their own Fitbit account (bring your own device); the **WEAR** study also gave Fitbits to some participants | participants share their Apple Health data from an iPhone (with or without an Apple Watch) |
| **Where** | the main CDR v9 dataset, like everything else | a **separate** CDR v9 dataset, released "off-cycle" in September 2026, joined by `person_id`. Not in the Data Explorer |
| **People with steps** | about **68,500** | about **18,000** |
| **Layout** | **wide**: one column per measure (`steps`, `sedentary_minutes`, …) | **long**: one row per measurement, with `data_type`, `value`, and `units` (**2.2**) |
| **Detail** | **daily** summary tables, and **minute-level** ("intraday") tables | individual measurements: every few seconds to every few hours; no daily summary |
| **Also has** | heart rate zones, sleep stages, scales | resting heart rate, heart rate variability, blood pressure, **blood glucose** (about 1,000 people), walking speed, VO2 max |

### Wear time and valid days

A day with **300 steps** could mean a very inactive day, or a device left on the charger. Counting un-worn days as inactive makes people look less active than they are, and makes people who wear their device less look the least active.

So studies keep only **valid days**: days the device was clearly worn. A common rule for Fitbit, used in published *All of Us* step-count studies:

| Rule | Why |
|---|---|
| at least **10 hours of wear** (600 minutes), judged from **heart-rate** minutes | Fitbit only records heart rate when it's on a wrist |
| at least **100 steps** | fewer means barely worn |
| at most **45,000 steps** | more is likely an error (or a device shaken on a bike) |

Then people need **enough** valid days (e.g. 10 or more) for their average to mean something.

### One value per person

Wearables give hundreds of values per person. Most analyses need **one**: e.g. the **median daily steps** over valid days. The median, not the mean, so one marathon day doesn't dominate.

### Steps from different devices aren't the same measurement

| Device | Worn | Misses |
|---|---|---|
| Fitbit | on the wrist, all day (often at night too) | little, when worn |
| Apple Watch | on the wrist | little, when worn |
| **iPhone** | in a pocket or bag, **when carried** | every step taken without the phone (at home, at work) |

An iPhone usually records **fewer** steps than a watch for the same person. Pooling them as one "steps" variable would mix **device** with **activity**.

### Who has wearable data? (No weights, again)

Owning a Fitbit or an iPhone and choosing to share its data isn't random: it depends on age, income, health, and interest in fitness. As with the surveys (**5.1**, *No survey weights*), results describe **people with wearable data**, not all participants, and certainly not the whole country.

### Cost: big tables, and clustering

BigQuery charges for the **data a query reads** ($6.25 per TB). The minute-level Fitbit tables are over **6 TB each**. Two things keep them affordable:

- They're **clustered by `person_id`**: each person's rows are stored together. A query that filters on `person_id` reads only those people's blocks.
- So **always** filter them by `person_id`, and keep the list of people short.

A **dry run** (free) estimates a query's cost before running it, including the savings from clustering. In our test run, the same minute-level query cost about **$0.03 for 50 people** and **$19 for 10,000**: the cost grows with the number of people. We also check what each query **actually** billed, afterwards.

## Part 2. Setup

`aou_setup.R` has two cost helpers for this series:

| Helper | Does |
|---|---|
| `run_if_cheap(sql, max_dollars = 1)` | a free **dry run** first; runs the query only if it costs less than `max_dollars`. A 0-byte estimate usually means the filter matches nothing: those run only with `allow_unknown = TRUE` |
| `last_query_cost()` | what your most recent query **actually** billed |

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

cohort_sample <- load_cohort_sample()
cohort_sample %>% count(cohort)

ids_sql <- str_c(cohort_sample$person_id, collapse = ", ")   # don't print: participant IDs
HK <- HEALTHKIT_DATASET                                       # the Apple HealthKit dataset
HK

## Part 3. A tour of the tables, and their cost

`__TABLES__` lists every table in a dataset with its number of rows and size. Reading it is **free**.

In [ ]:
wearable_tables <- run_query(str_glue("
  SELECT table_id, row_count, ROUND(size_bytes / 1e9, 1) AS size_gb
  FROM `{CURRENT_DATASET}.__TABLES__`
  WHERE REGEXP_CONTAINS(table_id, r'^(activity_summary|heart_rate|sleep|steps|device$|wear_study)')
  UNION ALL
  SELECT CONCAT('healthkit: ', table_id), row_count, ROUND(size_bytes / 1e9, 1)
  FROM `{HK}.__TABLES__`")) %>%
  mutate(full_read_dollars = round(size_gb / 1000 * 6.25, 2)) %>%
  arrange(desc(size_gb))

wearable_tables

**🔍 Read the output:** each table, its size, and what reading **all** of it would cost (`full_read_dollars`). In CDR v9:

| Table | Size | One row per… |
|---|---|---|
| `heart_rate_intraday`, `steps_intraday` | over **6,000 GB** each | person per **minute** (Fitbit) |
| `sleep_level` | about 75 GB | person per sleep stage (Fitbit) |
| `heart_rate_summary` | about 19 GB | person per day per heart-rate **zone** (Fitbit) |
| `activity_summary` | about 9 GB | person per **day** (Fitbit): steps, active minutes, calories |
| `sleep_daily_summary` | about 6 GB | person per night (Fitbit) |
| `device`, `wear_study` | small | Fitbit devices; WEAR study consent |
| `healthkit: activity_measurements` | about 350 GB | one **measurement**: steps, distance, energy, … (Apple) |
| `healthkit: vital_measurements` | about 110 GB | one measurement: heart rate, blood pressure, glucose, … (Apple) |
| `healthkit: physical_measurements`, `device_and_source` | small | weight, height, …; which device recorded each measurement (Apple) |

`row_count` is `NA` for the two largest tables: they have more rows than R's whole numbers can hold (over 2 billion).

### What does a query cost?

A dry run reads nothing and costs nothing. Compare a daily table with a minute-level one:

In [ ]:
run_if_cheap(str_glue("SELECT person_id, date, steps FROM `{CURRENT_DATASET}.activity_summary`"),
             max_dollars = 0)

run_if_cheap(str_glue("SELECT person_id, datetime, steps FROM `{CURRENT_DATASET}.steps_intraday`
                       WHERE person_id IN ({ids_sql})"),
             max_dollars = 0)

**🔍 Read the output:** `max_dollars = 0` means "estimate only, never run".

- **`activity_summary`:** about 1.6 GB, a cent, in our test run. BigQuery reads only the **columns** you ask for, so 3 columns cost less than the whole table.
- **`steps_intraday`, for our 10,000 people:** about **3,000 GB, $19**. Clustering means BigQuery reads only these people's rows, but 10,000 people's minute-by-minute steps over years is still a lot. In Part 5 we use 100 people instead.

## Part 4. Fitbit: daily data, and valid days

One query, for our 10,000 people: daily steps and active minutes from `activity_summary`, plus each day's **wear minutes** from `heart_rate_summary`. Fitbit records minutes in each heart-rate **zone** (*Out of Range*, *Fat Burn*, *Cardio*, *Peak*), so their **sum** is the minutes the device recorded a heart rate: our measure of wear time.

`WITH wear AS (…)` adds up the zones per person per day; `LEFT JOIN` keeps every activity day, even without heart-rate data.

In [ ]:
fitbit_daily <- run_if_cheap(str_glue("
  WITH wear AS (
    SELECT person_id, date, SUM(minute_in_zone) AS wear_minutes
    FROM `{CURRENT_DATASET}.heart_rate_summary`
    WHERE person_id IN ({ids_sql})
    GROUP BY person_id, date
  )
  SELECT a.person_id, a.date, a.steps, a.sedentary_minutes, a.lightly_active_minutes,
         a.fairly_active_minutes, a.very_active_minutes, w.wear_minutes
  FROM `{CURRENT_DATASET}.activity_summary` AS a
  LEFT JOIN wear AS w ON a.person_id = w.person_id AND a.date = w.date
  WHERE a.person_id IN ({ids_sql})"), max_dollars = 1) %>%
  mutate(person_id = as.character(person_id))

last_query_cost()

fitbit_daily %>%
  summarise(people = n_distinct(person_id), days = n(),
            first_day = min(date), last_day = max(date))

**🔍 Read the output:** the dry-run estimate, the actual bill, and how much Fitbit data our sample has. In our test run: about 8.6 GB ($0.05); **926** of our 10,000 people had Fitbit data, about **850,000** person-days, from 2009 to 2024.

### What does a day look like?

Before setting a rule, look at the data: how are steps and wear minutes spread?

In [ ]:
fitbit_daily %>%
  summarise(across(c(steps, wear_minutes),
                   list(p01 = ~ quantile(.x, 0.01, na.rm = TRUE),
                        median = ~ median(.x, na.rm = TRUE),
                        p99 = ~ quantile(.x, 0.99, na.rm = TRUE),
                        missing = ~ round(100 * mean(is.na(.x)), 1)))) %>%
  pivot_longer(everything(), names_to = c("variable", "statistic"), names_sep = "_(?=[^_]+$)") %>%
  pivot_wider(names_from = statistic, values_from = value)

**🔍 Read the output:** the 1st percentile, median, and 99th percentile of daily steps and wear minutes, and the percent of days each is missing.

In our test run: median **5,142** steps a day; some days with 0 steps; and `wear_minutes` missing on **29%** of days.

- Days with **very few** steps are often days the device wasn't worn.
- `wear_minutes` is **missing** on days with no heart-rate data: older Fitbits without a heart-rate sensor, or the device wasn't worn at all.
- The median wear is **1,440 minutes**: a full day. Suspicious: does everyone wear their Fitbit 24 hours a day? We check this measure in Part 5.

### Apply the valid-day rule

In [ ]:
fitbit_daily <- fitbit_daily %>%
  mutate(valid_day = coalesce(!is.na(wear_minutes) & wear_minutes >= 600 &
                              between(steps, 100, 45000), FALSE))

fitbit_daily %>%
  summarise(days = n(),
            valid_days = sum(valid_day),
            pct_valid = round(100 * mean(valid_day), 1),
            median_steps_all_days = median(steps, na.rm = TRUE),
            median_steps_valid_days = median(steps[valid_day]))

`coalesce(…, FALSE)` turns `NA` into `FALSE`. Without it, a day with no step count gives `NA` (in R, `TRUE & NA` is `NA`), and `sum(valid_day)` becomes `NA` too.

**🔍 Read the output:** the share of days that pass the rule, and how median steps **change** when un-worn days are removed: usually **up**, by a lot. That's the bias that un-worn days create.

### ✍️ Your turn

Some studies use **8 hours** of wear (480 minutes) instead of 10. How many more valid days does that give, and does median daily steps change much?

<details>
<summary><b>Show answer</b></summary>

```r
fitbit_daily %>%
  mutate(valid_8h = !is.na(wear_minutes) & wear_minutes >= 480 & between(steps, 100, 45000)) %>%
  summarise(valid_10h = sum(valid_day), valid_8h = sum(valid_8h),
            median_10h = median(steps[valid_day]), median_8h = median(steps[valid_8h]))
```

The looser rule keeps more days, and the median drops a little: partly-worn days have fewer steps. Like the coding choices in 5.2, a rule like this is worth testing both ways.

</details>

In [ ]:
# Your code here

### One value per person

For each person: the number of valid days, and the **median** steps over them. We keep people with at least **10** valid days.

In [ ]:
fitbit_person <- fitbit_daily %>%
  filter(valid_day) %>%
  group_by(person_id) %>%
  summarise(fitbit_valid_days    = n(),
            fitbit_median_steps  = median(steps),
            fitbit_first_day     = min(date),
            fitbit_last_day      = max(date)) %>%
  filter(fitbit_valid_days >= 10)

fitbit_person %>%
  summarise(people = n(),
            median_valid_days = median(fitbit_valid_days),
            median_of_median_steps = median(fitbit_median_steps))

save_data(fitbit_daily, "fitbit_daily.csv")

**🔍 Read the output:** how many people have at least 10 valid days, how many valid days a typical person has, and the typical person's median daily steps. In our test run: **840** people, a median of **390** valid days each (more than a year of data), and **5,569** median daily steps.

`fitbit_daily.csv` keeps every day, with `valid_day`, so **6.2** can choose days by **date** (e.g. only before a diagnosis).

## Part 5. Minute-level data, safely

Daily totals hide **when** people move. The minute-level table, `steps_intraday`, shows it, but it's over 6 TB. We query it for a **small** group: 100 people with Fitbit data (50 cases, 50 controls), adding up their steps by **hour of the day**, in BigQuery, so only 2 × 24 numbers come back.

In [ ]:
set.seed(2026)
hourly_people <- fitbit_person %>%
  left_join(cohort_sample %>% select(person_id, cohort), by = "person_id") %>%
  group_by(cohort) %>%
  slice_sample(n = 50) %>%
  ungroup()

hourly_sql <- function(ids) str_glue("
  SELECT EXTRACT(HOUR FROM SAFE_CAST(datetime AS DATETIME)) AS hour,
         SUM(steps) AS steps,
         COUNT(DISTINCT CONCAT(CAST(person_id AS STRING), CAST(DATE(SAFE_CAST(datetime AS DATETIME)) AS STRING))) AS person_days
  FROM `{CURRENT_DATASET}.steps_intraday`
  WHERE person_id IN ({str_c(ids, collapse = ', ')})
  GROUP BY hour")

hourly <- map(c("case", "control"), function(g) {
  ids <- hourly_people$person_id[hourly_people$cohort == g]
  run_if_cheap(hourly_sql(ids), max_dollars = 1) %>% mutate(cohort = g)
}) %>% list_rbind()

last_query_cost()

**🔍 Read the output:** the dry-run estimate for each group, and the actual cost of the last query. In our test run: about **5.7 GB, $0.04** per group of 50, matching the estimate: a tiny fraction of the 6,000 GB table, thanks to clustering by `person_id`.

Now average steps in each hour, per person-day, by group:

In [ ]:
hourly <- hourly %>%
  filter(!is.na(hour)) %>%
  mutate(steps_per_person_day = steps / person_days)

ggplot(hourly, aes(x = hour, y = steps_per_person_day, colour = cohort)) +
  geom_line(linewidth = 1) +
  geom_point(size = 1.5) +
  scale_x_continuous(breaks = seq(0, 23, 3)) +
  labs(x = "Hour of the day", y = "Average steps in that hour",
       colour = NULL, title = "When people move: steps by hour of the day",
       subtitle = "50 cases and 50 controls with Fitbit data, all their days") +
  theme_minimal(base_size = 14)

**🔍 Read the output:** steps rise from about 5 am, stay high through the day, and fall in the evening, in both groups. In our test run, controls took clearly **more** steps from about 8 am to 6 pm (about 400 to 450 an hour, vs 330 to 360), while the groups were similar at night.

These are **all** recorded days, not only valid days, and only 100 people: a picture, not a result. (`person_days` counts days with **any** step recorded in that hour, so it's an approximation.)

⚠️ Never plot **one** person's minute-by-minute steps outside the Workbench: that's participant-level data.

### Is our wear measure any good?

In Part 4 the median day had **1,440 wear minutes**: exactly 24 hours. The zone total is cheap, but is it really wear time? Published *All of Us* step studies count the **minutes with a heart-rate reading** in the minute-level table instead. Compare the two, for the same 100 people (about $0.04):

In [ ]:
check_ids <- hourly_people$person_id

hr_minutes <- run_if_cheap(str_glue("
  SELECT person_id,
         DATE(SAFE_CAST(datetime AS DATETIME)) AS date,
         COUNT(DISTINCT DATETIME_TRUNC(SAFE_CAST(datetime AS DATETIME), MINUTE)) AS hr_minutes
  FROM `{CURRENT_DATASET}.heart_rate_intraday`
  WHERE person_id IN ({str_c(check_ids, collapse = ', ')})
  GROUP BY 1, 2"), max_dollars = 1) %>%
  mutate(person_id = as.character(person_id))

last_query_cost()

fitbit_daily %>%
  filter(person_id %in% check_ids) %>%
  inner_join(hr_minutes, by = c("person_id", "date")) %>%
  summarise(days                 = n(),
            median_zone_minutes  = median(wear_minutes, na.rm = TRUE),
            median_hr_minutes    = median(hr_minutes),
            pct_zone_is_1440     = round(100 * mean(wear_minutes == 1440, na.rm = TRUE), 1),
            pct_zone_600_or_more = round(100 * mean(wear_minutes >= 600, na.rm = TRUE), 1),
            pct_hr_600_or_more   = round(100 * mean(hr_minutes >= 600), 1))

**🔍 Read the output:** in our test run:

| Wear measure | Median minutes | Days with 600+ minutes |
|---|---|---|
| heart-rate **zone total** (daily table, cheap) | 1,440 (exactly 24 hours on 61% of days) | 98.6% |
| minutes with a **heart-rate reading** (minute-level table) | 1,392 (about 23 hours) | 95.5% |

- These Fitbit users really do wear their devices almost around the clock, including at night (Fitbits track sleep).
- The zone total **overstates** wear: it often fills the day to exactly 1,440 minutes.
- At the 10-hour cutoff the two disagree on only about **3%** of days. So for this class, the cheap measure is good enough; for a paper, use the minute-level count (about $0.50 for our roughly 900 Fitbit users).

This is a habit worth keeping: **check a cheap measure against a better one, on a small group, before trusting it.**

## Part 6. Apple HealthKit

The HealthKit data is in a **different dataset** (`HK`), joined to ours by `person_id`. Its layout is **long**: every measurement is a row, and `data_type` says what it is. We want daily steps, so we:

1. keep `data_type = 'STEP_COUNT'`, for our people,
2. join `device_and_source` to see **which device** recorded each measurement (`product_type` starts with `iPhone` or `Watch`),
3. add up `value` per person, **day**, and **device**. `value` is stored as **text** (the same column also holds categories, such as sleep stages), so `SAFE_CAST(a.value AS FLOAT64)` turns it into a number first; anything that isn't a number becomes `NULL` instead of an error.

Step 3 keeps devices **separate**: when someone carries an iPhone **and** wears a Watch, both count the same steps, so adding them together would count steps **twice**.

💰 About $1: HealthKit tables aren't clustered by person, so BigQuery reads the needed columns of the whole table.

In [ ]:
healthkit_daily <- run_if_cheap(str_glue("
  SELECT a.person_id,
         DATE(SAFE_CAST(a.start_datetime AS TIMESTAMP)) AS date,
         CASE WHEN d.product_type LIKE 'Watch%' THEN 'Apple Watch'
              WHEN d.product_type LIKE 'iPhone%' THEN 'iPhone'
              ELSE 'other' END AS device,
         SUM(SAFE_CAST(a.value AS FLOAT64)) AS steps
  FROM `{HK}.activity_measurements` AS a
  LEFT JOIN `{HK}.device_and_source` AS d ON a.device_source_id = d.device_source_id
  WHERE a.data_type = 'STEP_COUNT'
    AND a.person_id IN ({ids_sql})
  GROUP BY 1, 2, 3"), max_dollars = 3) %>%
  mutate(person_id = as.character(person_id))

last_query_cost()

healthkit_daily %>%
  group_by(device) %>%
  summarise(people = n_distinct(person_id), person_days = n()) %>%
  mutate(across(c(people, person_days), ~ if_else(people <= 20, NA_integer_, .x)))   # hide small groups

**🔍 Read the output:** how many of our people have HealthKit steps, from each kind of device. In our test run: about **250** with iPhone steps and about **160** with Apple Watch steps; a few had other sources, too few to show (hidden as `NA`).

⚠️ Every table of counts you might share needs the small-count check, including quick ones like this.

### Do the devices agree?

HealthKit has no wear-time information, so we use a simpler rule: days with 100 to 45,000 steps. Then each person's **median** daily steps, per device:

In [ ]:
healthkit_person_device <- healthkit_daily %>%
  filter(between(steps, 100, 45000), device != "other") %>%
  group_by(person_id, device) %>%
  summarise(days = n(), median_steps = median(steps), .groups = "drop") %>%
  filter(days >= 10)

device_steps <- bind_rows(
  fitbit_person %>% transmute(person_id, device = "Fitbit", median_steps = fitbit_median_steps),
  healthkit_person_device %>% select(person_id, device, median_steps))

device_steps %>%
  group_by(device) %>%
  summarise(people = n(), median = median(median_steps),
            q1 = quantile(median_steps, 0.25), q3 = quantile(median_steps, 0.75)) %>%
  mutate(people = if_else(people <= 20, NA_integer_, people))

ggplot(device_steps, aes(x = device, y = median_steps, fill = device)) +
  geom_boxplot(outlier.shape = NA, show.legend = FALSE) +
  coord_cartesian(ylim = c(0, 20000)) +
  labs(x = NULL, y = "Each person's median daily steps",
       title = "Daily steps by device", subtitle = "People with at least 10 days") +
  theme_minimal(base_size = 14)

**🔍 Read the output:** typical daily steps by device. In our test run, each person's median daily steps were typically **5,569** with a Fitbit, **3,860** with an Apple Watch, and **2,223** with an iPhone.

- The **iPhone** records far fewer: it misses every step taken without it.
- The **Watch** is below the Fitbit, but these are **different people**, who differ in more than their device. The within-person comparison below is fairer.

So in **6.2** we analyse Fitbit steps, and treat Apple steps as a separate measure, never pooled with Fitbit.

The boxes hide individual people: each box shows the middle half of people (the IQR), and the line inside it the median.

### People with both an iPhone and a Watch

The fairest comparison is **within** the same people:

In [ ]:
healthkit_person_device %>%
  select(person_id, device, median_steps) %>%
  pivot_wider(names_from = device, values_from = median_steps) %>%
  filter(!is.na(iPhone), !is.na(`Apple Watch`)) %>%
  summarise(people = n(),
            median_iphone = median(iPhone),
            median_watch  = median(`Apple Watch`),
            median_ratio_iphone_to_watch = round(median(iPhone / `Apple Watch`), 2)) %>%
  mutate(people = if_else(people <= 20, NA_integer_, people))

**🔍 Read the output:** for people with both devices, the typical ratio of iPhone steps to Watch steps. A ratio below 1 means the phone records fewer steps than the watch, for the **same** person.

## Part 7. One row per person; who has wearables?

In [ ]:
wearables_person <- cohort_sample %>%
  select(person_id, cohort, age, sex_at_birth) %>%
  left_join(fitbit_person, by = "person_id") %>%
  left_join(healthkit_person_device %>%
              mutate(device = recode(device, "Apple Watch" = "watch", "iPhone" = "iphone")) %>%
              select(person_id, device, median_steps) %>%
              pivot_wider(names_from = device, values_from = median_steps,
                          names_prefix = "healthkit_steps_"),
            by = "person_id") %>%
  mutate(has_fitbit    = !is.na(fitbit_median_steps),
         has_healthkit = person_id %in% healthkit_daily$person_id)

nrow(wearables_person)   # still 10,000
save_data(wearables_person, "wearables_person.csv")

wearables_person %>%
  group_by(cohort) %>%
  summarise(people = n(),
            pct_fitbit    = round(100 * mean(has_fitbit), 1),
            pct_healthkit = round(100 * mean(has_healthkit), 1),
            pct_either    = round(100 * mean(has_fitbit | has_healthkit), 1))

**🔍 Read the output:** the percent of cases and controls with **usable** Fitbit data (10+ valid days) and with any HealthKit steps. In our test run: Fitbit **7.3%** of cases and **9.5%** of controls; HealthKit **1.8%** and **3.3%**. Only about 1 in 10 of our sample has either, and cases less often.

### Who has a Fitbit?

In [ ]:
wearables_person %>%
  group_by(has_fitbit) %>%
  summarise(people     = n(),
            mean_age   = round(mean(age), 1),
            pct_female = round(100 * mean(sex_at_birth == "Female", na.rm = TRUE), 1),
            pct_case   = round(100 * mean(cohort == "case"), 1))

**🔍 Read the output:** in our test run, people with Fitbit data were a little **younger** (mean age 58 vs 59), more often **female** (63% vs 59%), and less often **cases** (43% vs 51%). Wearable users are a **selected** group.

**💬 Discuss:**

1. Why might people with type 2 diabetes be **less** (or more) likely to have Fitbit data?
2. A result like "people with diabetes take fewer steps" is about **Fitbit users** with and without diabetes. Who might it not apply to?
3. Fitbit data starts in 2008 for some people, and only recently for others. Why could **when** someone's data starts matter? (**6.2**: steps before or after a diagnosis.)

### ✍️ Your turn

Compare people **with and without HealthKit data** the same way. Are they different from Fitbit users?

<details>
<summary><b>Show answer</b></summary>

```r
wearables_person %>%
  group_by(has_healthkit) %>%
  summarise(people = n(), mean_age = round(mean(age), 1),
            pct_female = round(100 * mean(sex_at_birth == "Female", na.rm = TRUE), 1),
            pct_case = round(100 * mean(cohort == "case"), 1))
```

Apple data needs an iPhone, which isn't the same group as Fitbit owners. Each device's users are selected in their **own** way.

</details>

In [ ]:
# Your code here

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| *The dry run says this reads 0 bytes … Not run* | the filter matches nothing (e.g. wrong IDs) | check the IDs; `allow_unknown = TRUE` runs it anyway |
| *Not run: over the limit* | the query reads more than `max_dollars` | select fewer columns, filter more, or raise the limit **knowingly** |
| *Not found: Dataset … offcycle_healthkit* | the HealthKit dataset isn't available to this workspace | check `HEALTHKIT_DATASET` in `aou_setup.R`, and your workspace's data access |
| A query on `steps_intraday` is slow or costly | it isn't filtered by `person_id`, or the list is very long | filter by `person_id`, with fewer people |
| `row_count` is `NA` | more than 2 billion rows | read `size_gb` instead |
| *No matching signature for aggregate function SUM … STRING* | HealthKit's `value` column is text | `SUM(SAFE_CAST(value AS FLOAT64))` |

## ✅ Key takeaways

- *All of Us* has **Fitbit** data (main dataset; about 68,500 people) and **Apple HealthKit** data (a separate dataset; about 18,000)
- Price queries **before** running them; check what they **actually** cost after. Minute-level tables are over 6 TB: always filter by `person_id`
- Keep only **valid days**, then summarise to **one value per person** (the median)
- **iPhone, Apple Watch, and Fitbit** steps measure different things: don't pool them
- People with wearable data are a **selected** group: results describe them

**Next:** **6.2 steps_and_t2d**: daily steps and type 2 diabetes, using only steps measured **before** a diagnosis.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).

## 🧑‍🏫 Instructor notes

- **Start with the cost table** (Part 3): "what would one careless query cost?" gets everyone's attention.
- **The valid-day rule** is the key idea: show how median steps change when un-worn days are dropped before explaining why.
- **iPhone vs Watch:** ask students where their phone is right now. Most aren't carrying it.
- **Privacy:** plots of one person's minute-level data are participant-level data. Only show groups.
- **Likely question:** "Can I get Apple and Fitbit data for the same person?" A few people have both; compare within them, never pool.